# Huấn luyện mô hình Abalone

Notebook này triển khai bước `Build Models` và `Evaluate Models` trên tập huấn luyện. Toàn bộ 10 thuật toán hồi quy bắt buộc sẽ được đánh giá bằng `5-fold Cross-validation` theo đúng yêu cầu đề bài.

## 1. Mục tiêu của bước huấn luyện

- Sử dụng các tập dữ liệu đã tiền xử lý ở bước 02.
- Chạy đủ 10 mô hình hồi quy baseline.
- Đánh giá bằng `5-fold Cross-validation` trên tập train.
- Ghi nhận các chỉ số `MSE`, `RMSE`, `RSE`, `MAE` và `execution_time`.
- Lưu kết quả và mô hình baseline để phục vụ bước đánh giá và tối ưu hóa sau đó.

## 2. Chuẩn bị thư viện

In [1]:
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time

import joblib
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor

# Dung helper metric va CV tu src/models/evaluate.py
from src.models.evaluate import build_kfold, run_cv_predictions

warnings.filterwarnings('ignore')


## 3. Load các tập dữ liệu đã tiền xử lý

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_MODELS_DIR = PROJECT_ROOT / 'outputs' / 'models'
OUTPUT_METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'

OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)

train_encoded_df = pd.read_csv(PROCESSED_DIR / 'abalone_train_encoded.csv')
train_standard_df = pd.read_csv(PROCESSED_DIR / 'abalone_train_standard.csv')
train_robust_df = pd.read_csv(PROCESSED_DIR / 'abalone_train_robust_log.csv')

display(train_encoded_df.head())

,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,1.0,0.0,0.0,0.525,0.430,0.135,0.8435,0.4325,0.1800,0.1815,9
1,0.0,1.0,0.0,0.430,0.325,0.100,0.3645,0.1575,0.0825,0.1050,7
2,0.0,0.0,1.0,0.455,0.350,0.105,0.4160,0.1625,0.0970,0.1450,11
3,0.0,0.0,1.0,0.205,0.155,0.045,0.0425,0.0170,0.0055,0.0155,7
4,1.0,0.0,0.0,0.590,0.465,0.160,1.1005,0.5060,0.2525,0.2950,13


## 4. Chuẩn bị tập train cho từng nhóm mô hình

- `encoded_only` dùng cho nhóm mô hình cây/ensemble.
- `standard_scaled` dùng cho nhóm mô hình nhạy với thang đo.
- `robust_log_scaled` sẽ được ưu tiên để kiểm tra ở bước tối ưu hóa, không dùng cho baseline ở notebook này.

In [3]:
target_col = 'Rings'

X_train_encoded = train_encoded_df.drop(columns=[target_col])
y_train_encoded = train_encoded_df[target_col]

X_train_standard = train_standard_df.drop(columns=[target_col])
y_train_standard = train_standard_df[target_col]

prepared_datasets = {
    'encoded_only': (X_train_encoded, y_train_encoded), # Đây là dataset đã được one-hot encoding và label encoding, nhưng chưa được chuẩn hóa.
    'standard_scaled': (X_train_standard, y_train_standard), # Đây là dataset đã được chuẩn hóa bằng StandardScaler, nhưng chưa được one-hot encoding hoặc label encoding.
}

for name, (X_part, y_part) in prepared_datasets.items():
    print(name, X_part.shape, y_part.shape)

encoded_only (2923, 10) (2923,)
standard_scaled (2923, 10) (2923,)


## 5. Khai báo 10 mô hình hồi quy baseline bắt buộc

In [4]:
baseline_models = {
    'LinearRegression': LinearRegression(),
    'KNeighborsRegressor': KNeighborsRegressor(),
    'RidgeRegression': Ridge(),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42, n_jobs=-1),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42),
    'SGDRegressor': SGDRegressor(random_state=42, max_iter=2000, tol=1e-3),
    'SVR': SVR(),
    'LinearSVR': LinearSVR(random_state=42, max_iter=10000),
    'MLPRegressor': MLPRegressor(random_state=42, max_iter=1000),
}

dataset_mapping = {
    'LinearRegression': 'standard_scaled',
    'KNeighborsRegressor': 'standard_scaled',
    'RidgeRegression': 'standard_scaled',
    'DecisionTreeRegressor': 'encoded_only',
    'RandomForestRegressor': 'encoded_only',
    'GradientBoostingRegressor': 'encoded_only',
    'SGDRegressor': 'standard_scaled',
    'SVR': 'standard_scaled',
    'LinearSVR': 'standard_scaled',
    'MLPRegressor': 'standard_scaled',
}

display(pd.DataFrame({'model': list(dataset_mapping.keys()), 'dataset_version': list(dataset_mapping.values())}))

,model,dataset_version
0,LinearRegression,standard_scaled
1,KNeighborsRegressor,standard_scaled
2,RidgeRegression,standard_scaled
3,DecisionTreeRegressor,encoded_only
4,RandomForestRegressor,encoded_only
5,GradientBoostingRegressor,encoded_only
6,SGDRegressor,standard_scaled
7,SVR,standard_scaled
8,LinearSVR,standard_scaled
9,MLPRegressor,standard_scaled


## 6. Hàm tính metric và hàm huấn luyện/evaluate

In [5]:
# Dung run_cv_predictions() tu src/models/evaluate.py de lay du doan OOF va metric
def run_cv_and_train(model_name, model, data_key, cv_strategy):
    X_current, y_current = prepared_datasets[data_key]
    start_time = time.perf_counter()
    y_oof_pred, metrics = run_cv_predictions(model, X_current, y_current, cv_strategy)
    final_model = clone(model)
    final_model.fit(X_current, y_current)
    elapsed = time.perf_counter() - start_time
    return X_current, y_current, y_oof_pred, final_model, elapsed, metrics


## 7. Thiết lập 5-fold Cross-validation

In [6]:
# Dung build_kfold() tu src/models/evaluate.py
cv = build_kfold(n_splits=5, shuffle=True, random_state=42)
print(cv)


KFold(n_splits=5, random_state=42, shuffle=True)


## 8. Huấn luyện và đánh giá toàn bộ 10 mô hình baseline

In [7]:
results = []
# DataFrame để lưu trữ dự đoán OOF của tất cả các mô hình, giúp chúng ta có thể so sánh trực tiếp
#  giữa các mô hình trên cùng một tập dữ liệu. Cột 'y_true' sẽ chứa giá trị thực tế, và 
# các cột tiếp theo sẽ chứa dự đoán OOF của từng mô hình tương ứng. 
# Chúng ta sẽ reset index của y_train_encoded để đảm bảo rằng nó khớp với index của dự đoán OOF,
#  sau đó sắp xếp lại index để đảm bảo rằng các giá trị được so khớp đúng với nhau khi thêm 
# vào DataFrame oof_predictions. Điều này giúp chúng ta dễ dàng so sánh hiệu suất của các mô hình 
# khác nhau trên cùng một tập dữ liệu và trực quan hóa sự khác biệt giữa chúng.
oof_predictions = pd.DataFrame({'y_true': y_train_encoded.reset_index(drop=True)})
trained_model_artifacts = {}

for model_name, model in baseline_models.items():
    data_key = dataset_mapping[model_name]
    X_current, y_current, y_oof_pred, final_model, elapsed, metrics = run_cv_and_train(
        model_name,
        model,
        data_key,
        cv,
    )

    results.append(
        {
            'model': model_name,
            'dataset_version': data_key,
            'mse': metrics['mse'],
            'rmse': metrics['rmse'],
            'rse': metrics['rse'],
            'mae': metrics['mae'],  
            'execution_time_sec': elapsed,
        }
    )

    oof_predictions[model_name] = pd.Series(y_oof_pred, index=y_current.index).sort_index().values

    trained_model_artifacts[model_name] = {
        'model': final_model,
        'dataset_version': data_key,
        'feature_names': X_current.columns.tolist(),
    }

results_df = pd.DataFrame(results).sort_values(by='rmse', ascending=True).reset_index(drop=True)
display(results_df.round(4))

,model,dataset_version,mse,rmse,rse,mae,execution_time_sec
0,GradientBoostingRegressor,encoded_only,4.7163,2.1717,0.4497,1.5304,1.3256
1,SVR,standard_scaled,4.7549,2.1806,0.4534,1.4823,1.4044
2,MLPRegressor,standard_scaled,4.9047,2.2146,0.4677,1.4979,7.4329
3,RandomForestRegressor,encoded_only,4.9276,2.2198,0.4699,1.5640,2.3422
4,LinearRegression,standard_scaled,5.1175,2.2622,0.4880,1.6026,5.2685
5,RidgeRegression,standard_scaled,5.1189,2.2625,0.4881,1.6020,0.0295
6,SGDRegressor,standard_scaled,5.2445,2.2901,0.5001,1.6114,0.0493
7,LinearSVR,standard_scaled,5.2501,2.2913,0.5007,1.5672,0.0466
8,KNeighborsRegressor,standard_scaled,5.4723,2.3393,0.5218,1.6213,3.4916
9,DecisionTreeRegressor,encoded_only,9.1150,3.0191,0.8692,2.0859,0.1312


## 9. Xếp hạng nhanh theo từng metric

In [8]:
top_by_rmse = results_df.nsmallest(3, 'rmse')[['model', 'rmse', 'dataset_version']]
top_by_mae = results_df.nsmallest(3, 'mae')[['model', 'mae', 'dataset_version']]
top_by_mse = results_df.nsmallest(3, 'mse')[['model', 'mse', 'dataset_version']]
top_by_rse = results_df.nsmallest(3, 'rse')[['model', 'rse', 'dataset_version']]

print('Top 3 theo RMSE:')
display(top_by_rmse.round(4))
print('Top 3 theo MAE:')
display(top_by_mae.round(4))
print('Top 3 theo MSE:')
display(top_by_mse.round(4))
print('Top 3 theo RSE:')
display(top_by_rse.round(4))

Top 3 theo RMSE:


,model,rmse,dataset_version
0,GradientBoostingRegressor,2.1717,encoded_only
1,SVR,2.1806,standard_scaled
2,MLPRegressor,2.2146,standard_scaled


Top 3 theo MAE:


,model,mae,dataset_version
1,SVR,1.4823,standard_scaled
2,MLPRegressor,1.4979,standard_scaled
0,GradientBoostingRegressor,1.5304,encoded_only


Top 3 theo MSE:


,model,mse,dataset_version
0,GradientBoostingRegressor,4.7163,encoded_only
1,SVR,4.7549,standard_scaled
2,MLPRegressor,4.9047,standard_scaled


Top 3 theo RSE:


,model,rse,dataset_version
0,GradientBoostingRegressor,0.4497,encoded_only
1,SVR,0.4534,standard_scaled
2,MLPRegressor,0.4677,standard_scaled


## 10. Lưu kết quả baseline

In [9]:
results_df.to_csv(OUTPUT_METRICS_DIR / 'baseline_cv_results.csv', index=False)
oof_predictions.to_csv(OUTPUT_METRICS_DIR / 'baseline_oof_predictions.csv', index=False)

for model_name, artifact in trained_model_artifacts.items():
    joblib.dump(artifact, OUTPUT_MODELS_DIR / f'baseline_{model_name}.joblib')

training_metadata = {
    'cv_strategy': {'type': 'KFold', 'n_splits': 5, 'shuffle': True, 'random_state': 42},
    'metrics': ['mse', 'rmse', 'rse', 'mae', 'execution_time_sec'],
    'dataset_mapping': dataset_mapping,
    'results_file': 'outputs/metrics/baseline_cv_results.csv',
    'oof_predictions_file': 'outputs/metrics/baseline_oof_predictions.csv',
}

with open(OUTPUT_METRICS_DIR / 'baseline_training_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print('Đã lưu kết quả baseline và model artifacts.')

Đã lưu kết quả baseline và model artifacts.


## 11. Kết luận bước huấn luyện baseline

- Đã chạy đủ 10 thuật toán hồi quy bắt buộc trên tập train.
- Việc đánh giá được thực hiện bằng `5-fold Cross-validation` đúng theo yêu cầu đề bài.
- Các mô hình được huấn luyện trên phiên bản dữ liệu phù hợp với bản chất thuật toán, thay vì dùng một cách tiền xử lý duy nhất cho tất cả.
- Kết quả baseline này sẽ là mốc so sánh để thực hiện tuning và optimization ở notebook bước 05.